In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [3]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [11]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

CLS 8

In [13]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_8/embeddings_cls_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_8/embeddings_cls_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.06714
test f1: 0.06408
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.6,0.507042,0.601876,0.550404,0.511684,0.607177,0.555355


In [14]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_32/embeddings_cls_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_32/embeddings_cls_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.08524
test f1: 0.0799
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.6,0.507042,0.601876,0.550404,0.511684,0.607177,0.555355


In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_128/embeddings_cls_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_128/embeddings_cls_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.0958
test f1: 0.088
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.6,0.507042,0.601876,0.550404,0.511684,0.607177,0.555355


In [37]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_320/embeddings_cls_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_320/embeddings_cls_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.09839
test f1: 0.08847
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.6,0.507042,0.601876,0.550404,0.511684,0.607177,0.555355


Layer Mean

In [16]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_8/embeddings_layer_mean_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_8/embeddings_layer_mean_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07027
test f1: 0.06763
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


In [17]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_16/embeddings_layer_mean_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_16/embeddings_layer_mean_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08883
test f1: 0.08456
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 2
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.549451,0.473934,0.508906
1,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


In [18]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_32/embeddings_layer_mean_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_32/embeddings_layer_mean_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.1037
test f1: 0.09826
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 3
Survival rate: 0.6


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,23,0.8,0.584270,0.506083,0.542373,0.517906,0.505376,0.511565
2,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.549451,0.473934,0.508906
3,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


In [19]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_64/embeddings_layer_mean_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_64/embeddings_layer_mean_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.11772
test f1: 0.11208
Validation pairs with F1 > 0.5: 9
Those also with test F1 > 0.5: 7
Survival rate: 0.7777777777777778


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,33,0.8,0.985782,0.506083,0.668810,0.994565,0.491935,0.658273
1,heme,33,0.8,0.971564,0.488095,0.649762,0.961957,0.460938,0.623239
3,1.14,33,0.8,0.810427,0.422222,0.555195,0.842391,0.411141,0.552585
5,ig-like,46,0.6,0.542857,0.492925,0.516687,0.535000,0.520681,0.527744
4,GO:0020037,33,0.8,0.985782,0.382353,0.550993,0.994565,0.353282,0.521368
6,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.549451,0.473934,0.508906
7,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


In [20]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_128/embeddings_layer_mean_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_128/embeddings_layer_mean_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.12895
test f1: 0.12182
Validation pairs with F1 > 0.5: 10
Those also with test F1 > 0.5: 8
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,GO:0005506,33,0.8,0.985782,0.506083,0.668810,0.994565,0.491935,0.658273
2,heme,33,0.8,0.971564,0.488095,0.649762,0.961957,0.460938,0.623239
0,nudix hydrolase,109,0.8,1.000000,0.563636,0.720930,1.000000,0.431373,0.602740
4,1.14,33,0.8,0.810427,0.422222,0.555195,0.842391,0.411141,0.552585
6,ig-like,46,0.6,0.542857,0.492925,0.516687,0.535000,0.520681,0.527744
5,GO:0020037,33,0.8,0.985782,0.382353,0.550993,0.994565,0.353282,0.521368
7,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.549451,0.473934,0.508906
8,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


In [38]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_320/embeddings_layer_mean_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_320/embeddings_layer_mean_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.13937
test f1: 0.13055
Validation pairs with F1 > 0.5: 12
Those also with test F1 > 0.5: 10
Survival rate: 0.8333333333333334


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,217,0.8,0.923077,0.813559,0.864865,0.823529,0.807692,0.815534
2,GO:0005506,33,0.8,0.985782,0.506083,0.668810,0.994565,0.491935,0.658273
3,heme,33,0.8,0.971564,0.488095,0.649762,0.961957,0.460938,0.623239
1,nudix hydrolase,109,0.8,1.000000,0.563636,0.720930,1.000000,0.431373,0.602740
10,c-type lectin,170,0.8,0.545455,0.488372,0.515337,0.600000,0.542169,0.569620
5,1.14,33,0.8,0.810427,0.422222,0.555195,0.842391,0.411141,0.552585
7,ig-like,46,0.6,0.542857,0.492925,0.516687,0.535000,0.520681,0.527744
6,GO:0020037,33,0.8,0.985782,0.382353,0.550993,0.994565,0.353282,0.521368
8,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.549451,0.473934,0.508906
9,transmembrane,1,0.5,0.999506,0.347475,0.515677,0.999496,0.339904,0.507291


max

In [21]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_8/embeddings_max_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_8/embeddings_max_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07827
test f1: 0.07604
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
1,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748


In [23]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_16/embeddings_max_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_16/embeddings_max_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.09066
test f1: 0.0877
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
1,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748


In [24]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_32/embeddings_max_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_32/embeddings_max_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.11179
test f1: 0.1066
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
1,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748


In [25]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_64/embeddings_max_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_64/embeddings_max_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.12265
test f1: 0.11612
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
1,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748


In [26]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_128/embeddings_max_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_128/embeddings_max_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.1362
test f1: 0.12774
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 7
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,79,0.6,0.543373,0.893069,0.675655,0.556904,0.880365,0.682236
1,GO:0106310,79,0.6,0.448795,0.926617,0.604708,0.452340,0.922261,0.606977
2,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
3,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748
4,GO:0061630,65,0.6,0.499154,0.652655,0.565676,0.503831,0.650990,0.568035
5,GO:0004674,79,0.6,0.392169,0.873826,0.541372,0.407279,0.879052,0.556652
6,2.7,79,0.6,0.536145,0.545677,0.540869,0.551127,0.559531,0.555297


In [39]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_320/embeddings_max_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_320/embeddings_max_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.14101
test f1: 0.13046
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 7
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,79,0.6,0.543373,0.893069,0.675655,0.556904,0.880365,0.682236
1,GO:0106310,79,0.6,0.448795,0.926617,0.604708,0.452340,0.922261,0.606977
2,GO:0003924,6,0.6,0.432287,0.879562,0.579675,0.435500,0.856870,0.577492
3,GO:0005525,6,0.6,0.478027,0.708777,0.570969,0.481086,0.707561,0.572748
4,GO:0061630,65,0.6,0.499154,0.652655,0.565676,0.503831,0.650990,0.568035
5,GO:0004674,79,0.6,0.392169,0.873826,0.541372,0.407279,0.879052,0.556652
6,2.7,79,0.6,0.536145,0.545677,0.540869,0.551127,0.559531,0.555297


Mean

In [27]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_8/embeddings_mean_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_8/embeddings_mean_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.06792
test f1: 0.06593
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823


In [28]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_16/embeddings_mean_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_16/embeddings_mean_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08009
test f1: 0.07615
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823


In [29]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_32/embeddings_mean_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_32/embeddings_mean_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.09324
test f1: 0.08802
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823


In [30]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_64/embeddings_mean_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_64/embeddings_mean_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.11082
test f1: 0.10288
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 2
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,62,0.8,0.891892,0.600000,0.717391,0.935484,0.568627,0.707317
1,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823


In [31]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_128/embeddings_mean_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_128/embeddings_mean_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.12309
test f1: 0.1124
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 2
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,62,0.8,0.891892,0.600000,0.717391,0.935484,0.568627,0.707317
1,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823


In [40]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_320/embeddings_mean_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_320/embeddings_mean_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.13442
test f1: 0.12224
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 3
Survival rate: 0.6


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,62,0.8,0.891892,0.600000,0.717391,0.935484,0.568627,0.707317
2,GO:0005634,3,0.5,0.387001,0.868425,0.535406,0.388484,0.868425,0.536823
1,GO:0004252,141,0.6,0.602510,0.488136,0.539326,0.603376,0.467320,0.526703


Min

In [32]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_8/embeddings_min_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_8/embeddings_min_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.0656
test f1: 0.06182
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [33]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_16/embeddings_min_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_16/embeddings_min_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08138
test f1: 0.07765
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [34]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_32/embeddings_min_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_32/embeddings_min_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.09761
test f1: 0.09428
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,31,0.6,0.400639,0.744554,0.520956,0.402846,0.749772,0.524098
1,GO:0005634,30,0.5,0.379939,0.739840,0.502052,0.381575,0.745413,0.504763


In [35]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_64/embeddings_min_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_64/embeddings_min_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.1127
test f1: 0.10559
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 4
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,42,0.8,0.500000,0.661017,0.569343,0.441558,0.653846,0.527132
1,GO:0005506,49,0.8,0.739130,0.413625,0.530421,0.691630,0.422043,0.524207
2,protein kinase,31,0.6,0.400639,0.744554,0.520956,0.402846,0.749772,0.524098
4,GO:0005634,30,0.5,0.379939,0.739840,0.502052,0.381575,0.745413,0.504763


In [36]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_128/embeddings_min_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_128/embeddings_min_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.12018
test f1: 0.11025
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 4
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,42,0.8,0.500000,0.661017,0.569343,0.441558,0.653846,0.527132
1,GO:0005506,49,0.8,0.739130,0.413625,0.530421,0.691630,0.422043,0.524207
2,protein kinase,31,0.6,0.400639,0.744554,0.520956,0.402846,0.749772,0.524098
4,GO:0005634,30,0.5,0.379939,0.739840,0.502052,0.381575,0.745413,0.504763


In [41]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_320/embeddings_min_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_320/embeddings_min_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.12777
test f1: 0.1168
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 4
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,42,0.8,0.500000,0.661017,0.569343,0.441558,0.653846,0.527132
1,GO:0005506,49,0.8,0.739130,0.413625,0.530421,0.691630,0.422043,0.524207
2,protein kinase,31,0.6,0.400639,0.744554,0.520956,0.402846,0.749772,0.524098
4,GO:0005634,30,0.5,0.379939,0.739840,0.502052,0.381575,0.745413,0.504763
